# 三模型基线指标比较

读取 results/Regression、results/RandomForest、results/XGBOOST 下的 metrics.csv，不训练模型。

- 三模型复用 data/xgboost_features 中的训练、验证、测试特征，分类阈值固定为0.5。
- 请先运行 Regression.ipynb、RandomForest.ipynb 和 code/XGBOOST.ipynb，并执行各自末尾的指标导出单元格，再运行本笔记本。
- 当前已有旧 CSV 不代表本次统一特征实验，不能直接当作新结果使用。
- 按验证集 AUC 排列模型，展示测试集指标、三份样本 AUC 和比较图。
- 仅在笔记本内展示结果，不生成分析报告，不导出表格或图片。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

project_dir = Path("D:/credit")
model_names = {"Regression": "逻辑回归", "RandomForest": "随机森林", "XGBOOST": "XGBoost"}
all_metrics = pd.concat([
    pd.read_csv(project_dir / "results" / folder / "metrics.csv").assign(模型=name)
    for folder, name in model_names.items()
], ignore_index=True)

## 1. 验证集与测试集主要指标

先按验证集 AUC 从高到低排列模型，再按同一顺序展示测试集指标，不根据测试集重新选择参数。

In [ ]:
metric_columns = ["AUC", "KS", "Precision", "Recall", "F1"]
validation_comparison = (
    all_metrics.loc[all_metrics["样本"] == "验证集"]
    .set_index("模型")[metric_columns + ["样本数", "特征数", "阈值"]]
    .sort_values("AUC", ascending=False)
)
model_order = validation_comparison.index
test_comparison = (
    all_metrics.loc[all_metrics["样本"] == "测试集"]
    .set_index("模型")
    .loc[model_order, metric_columns + ["样本数", "特征数", "阈值", "结果来源"]]
)
print("验证集指标（按验证集 AUC 排序）")
display(validation_comparison.round(4))
print("测试集指标（沿用验证集排序）")
display(test_comparison.round(4))

## 2. 训练、验证与测试 AUC

三份样本的 AUC 并列展示，训练—验证差和训练—测试差仅作数值记录。

In [ ]:
auc_comparison = all_metrics.pivot(index="模型", columns="样本", values="AUC")
auc_comparison = auc_comparison.loc[model_order, ["训练集", "验证集", "测试集"]]
auc_comparison["训练-验证差"] = auc_comparison["训练集"] - auc_comparison["验证集"]
auc_comparison["训练-测试差"] = auc_comparison["训练集"] - auc_comparison["测试集"]
display(auc_comparison.round(4))

## 3. 测试集指标比较图

按验证集 AUC 排序，展示测试集区分指标和0.5阈值下的分类指标。不生成自动分析文字，不保存图片。

In [ ]:
plot_data = test_comparison.rename(index={"逻辑回归": "Logistic", "随机森林": "Random forest"})
figure, axes = plt.subplots(1, 2, figsize=(11, 4))
plot_data[["AUC", "KS"]].plot.bar(ax=axes[0], rot=0)
plot_data[["Precision", "Recall", "F1"]].plot.bar(ax=axes[1], rot=0)
axes[0].set(title="Test ranking performance", xlabel="", ylabel="Score", ylim=(0, 1))
axes[1].set(title="Test classification at threshold 0.5", xlabel="", ylabel="Score", ylim=(0, 1))
figure.tight_layout()
plt.show()